In [52]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('D:\Dev\Point-Cloud-Registration\source')

import pickle
import os
import gc
from copy import deepcopy
import pandas as pd
import numpy as np
from glob import glob
from pathlib import Path
import matplotlib.pyplot as plt
from multiprocessing import Pool

from cylinder_alignment.bundle_station import BundleStation
from cylinder_extraction.station import Station
from cylinder_alignment.station_alignment import StationAlignment
from cylinder_alignment.utils import extract_target_cylinder
from utils.write_output import write_array2txt
from utils.math_methods import calc_overlap_curve

import plotly.express as px

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [53]:
data_dir = Path('D:/REGISTRATION DATA/External/JB-567 27ft EXT Leica')
filename_template = '{}.e57'
ptc_col_names =  ["x","y","z"]

In [54]:
bundle = BundleStation(data_dir=data_dir)

# Read extracted cylinder data

In [55]:
summary_df = bundle.read_extracted_cylinder_data()

D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\Tank409 Scan_Plains_Tank_409-_Setup_001\Tank409 Scan_Plains_Tank_409-_Setup_001.pkl
D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\Tank409 Scan_Plains_Tank_409-_Setup_002\Tank409 Scan_Plains_Tank_409-_Setup_002.pkl
D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\Tank409 Scan_Plains_Tank_409-_Setup_003\Tank409 Scan_Plains_Tank_409-_Setup_003.pkl
D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\Tank409 Scan_Plains_Tank_409-_Setup_004\Tank409 Scan_Plains_Tank_409-_Setup_004.pkl
D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\Tank409 Scan_Plains_Tank_409-_Setup_005\Tank409 Scan_Plains_Tank_409-_Setup_005.pkl
D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\Tank409 Scan_Plains_Tank_409-_Setup_006\Tank409 Scan_Plains_Tank_409-_Setup_006.pkl
D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\Tank409 Scan_Plains_Tank_409-_Setup_007\Tank409 Scan_Plains_Tank_409-_Setup_007.pkl
D:\REGISTRATION DATA\External\JB-567 27ft EXT Le

In [56]:
px.scatter(
    summary_df[summary_df.radius < 50], y='radius', x='station', 
    color='station', hover_data=summary_df.columns
)

# Pairwise alignment

In [57]:
bundle.align_pairwise_station()

Align station Tank409 Scan_Plains_Tank_409-_Setup_001 and Tank409 Scan_Plains_Tank_409-_Setup_002:
3
calling pair matching
100 is larger than the chosen RMS_min threshold = 1
3.6729270786708073 is larger than the chosen RMS_min threshold = 1
100 is larger than the chosen RMS_min threshold = 1
100 is larger than the chosen RMS_min threshold = 1
3.708416157823505 is larger than the chosen RMS_min threshold = 1
[[ 86.           6.48858166  -4.29054451 -29.98394203   0.        ]
 [ 95.           9.1551609   26.25253296  -9.97262383   0.        ]
 [ 89.           5.29702663  -0.79474932  16.00209236   0.        ]]
[[ 48.           6.48423576 -12.43388844 -30.68794632   0.        ]
 [ 77.           9.15471077  22.30363083 -41.92332458   0.        ]
 [ 92.           5.52852726  25.75803566  -4.77010632   0.        ]]
RMS = 0.05016373679648871
86 48
95 77
89 92
Align station Tank409 Scan_Plains_Tank_409-_Setup_001 and Tank409 Scan_Plains_Tank_409-_Setup_003:
3
calling pair matching
3.676789809

In [58]:
bundle.remove_points()
with open(os.path.join(data_dir, 'rough_alignment.pkl'), 'wb') as f:
    pickle.dump(bundle, f)

In [59]:
with open(os.path.join(data_dir, 'rough_alignment.pkl'), 'rb') as f:
    bundle = pickle.load(f)

# Align all stations

## For not-connected stations which have only 2 cylinders
For each of these station: call it station i
- Determine the neighbour station j
- Analyse two possible matching between i and j
    - Apply the corresponding transformation matrix
    - Extract thin slices
    - Measure the overlap
    - The correct match is the transformation with the larger overlap

In [60]:
bundle.handle_uncertain(
    neighbor_info={
        '009': '008',
        '016': '015',
        '018': '017'
    }
)

In [61]:
bundle.align_all_stations_by_axis()

[{'name': 'Tank409 Scan_Plains_Tank_409-_Setup_002'}, {'name': 'Tank409 Scan_Plains_Tank_409-_Setup_004'}, {'name': 'Tank409 Scan_Plains_Tank_409-_Setup_005'}, {'name': 'Tank409 Scan_Plains_Tank_409-_Setup_001'}]
[{'name': 'Tank409 Scan_Plains_Tank_409-_Setup_003'}, {'name': 'Tank409 Scan_Plains_Tank_409-_Setup_001'}]
[{'name': 'Tank409 Scan_Plains_Tank_409-_Setup_004'}, {'name': 'Tank409 Scan_Plains_Tank_409-_Setup_005'}, {'name': 'Tank409 Scan_Plains_Tank_409-_Setup_001'}]
[{'name': 'Tank409 Scan_Plains_Tank_409-_Setup_005'}, {'name': 'Tank409 Scan_Plains_Tank_409-_Setup_001'}]
[{'name': 'Tank409 Scan_Plains_Tank_409-_Setup_006'}, {'name': 'Tank409 Scan_Plains_Tank_409-_Setup_001'}]
[{'name': 'Tank409 Scan_Plains_Tank_409-_Setup_007'}, {'name': 'Tank409 Scan_Plains_Tank_409-_Setup_004'}, {'name': 'Tank409 Scan_Plains_Tank_409-_Setup_005'}, {'name': 'Tank409 Scan_Plains_Tank_409-_Setup_001'}]
[{'name': 'Tank409 Scan_Plains_Tank_409-_Setup_008'}, {'name': 'Tank409 Scan_Plains_Tank_409-

In [63]:
bundle.cluster_cylinder()

In [64]:
bundle.remove_points()
with open(os.path.join(data_dir, 'globalization_phase_1.pkl'), 'wb') as f:
    pickle.dump(bundle, f)

In [65]:
with open(os.path.join(data_dir, 'globalization_phase_1.pkl'), 'rb') as f:
    bundle = pickle.load(f)

# Visual inspection

To identify group id that represents the target cylinder 

In [ ]:
fig = bundle.draw_cross_section()
fig.show()

# Cluster id of the target tank

In [ ]:
# np.mean(bundle.tanks_info['centers_xy'][bundle.tanks_info['centers_xy'].tank_label=='0'][['x', 'y']].values, axis=0)

array([ -4.30056797, -29.97789964])

In [ ]:
# bundle.tanks_info['centers_xy'][bundle.tanks_info['centers_xy'].tank_label=='0'][['x', 'y']].values

array([[ -4.29054451, -29.98394203],
       [ -4.29583584, -29.9930789 ],
       [ -4.28646451, -30.01101691],
       [ -4.30468802, -29.97069455],
       [ -4.29820809, -29.96607298],
       [ -4.28965075, -29.96128033],
       [ -4.31237721, -29.9782895 ],
       [ -4.32677479, -29.95882191]])

In [66]:
bundle.identify_target_tank()
print('target tank', bundle.target_tank_id)

span [0.5042802767222362, 0.5086976190795783, 0.46654465383746485, 0.9999207747444214]
target tank 3


In [4]:
bundle.set_target_tank(tank_id=3)

# Cylinder vertical axes alignment

Put all fragments of the target tank in one place. Include points at their foot

In [8]:
bundle.write_target_tank(
    write_dir=bundle.data_dir / 'target_tank' / 'local' / 'ground',
    load_data=True, extend_ground=True, transform=False)

Run the following cell to load original ground-extended data

In [10]:
for c in bundle.target_tank:
    c.update_filepath(new_dir=bundle.data_dir / 'target_tank' / 'local' / 'ground', num_keep_level=1, file_name=f'{Path(c.ptc_filepath).stem}.pcd')
    print(c.ptc_filepath)
                # c.read_points()

D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\target_tank\local\ground\Tank409 Scan_Plains_Tank_409-_Setup_004 cluster_138.pcd
D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\target_tank\local\ground\Tank409 Scan_Plains_Tank_409-_Setup_005 cluster_63.pcd
D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\target_tank\local\ground\Tank409 Scan_Plains_Tank_409-_Setup_006 cluster_76.pcd
D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\target_tank\local\ground\Tank409 Scan_Plains_Tank_409-_Setup_007 cluster_41.pcd
D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\target_tank\local\ground\Tank409 Scan_Plains_Tank_409-_Setup_008 cluster_73.pcd


# Calculate the alignment matrices for all target cylinders

In [11]:
bundle.calc_align_matrix(f'{bundle.target_tank_id}')

calculating alignment matrix for tank 3
Tank409 Scan_Plains_Tank_409-_Setup_004 cluster 138
Tank409 Scan_Plains_Tank_409-_Setup_005 cluster 63
Tank409 Scan_Plains_Tank_409-_Setup_006 cluster 76
Tank409 Scan_Plains_Tank_409-_Setup_007 cluster 41
Tank409 Scan_Plains_Tank_409-_Setup_008 cluster 73


In [21]:
bundle.remove_points()
with open(os.path.join(data_dir, 'rough_alignment.pkl'), 'wb') as f:
    pickle.dump(bundle, f)

In [22]:
with open(os.path.join(data_dir, 'rough_alignment.pkl'), 'rb') as f:
    bundle = pickle.load(f)

Transform all fragments into a (common) global coordinate and save them to file

In [12]:
bundle.write_target_tank(
    write_dir=bundle.data_dir / 'target_tank' / 'global' / 'ground', 
    load_data=True, transform=True, extend_ground=False, num_process=5)

# Thin slice

In [20]:
bundle.cut_horizontal(mode='station_cylinders', z_cut=3, thickness=0.01, write_dir=bundle.data_dir / 'target_tank' / 'global' / 'thin_slice')

Tank409 Scan_Plains_Tank_409-_Setup_004 cluster 138
write to D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\target_tank\global\thin_slice\Tank409 Scan_Plains_Tank_409-_Setup_004.pcd
Tank409 Scan_Plains_Tank_409-_Setup_005 cluster 63
write to D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\target_tank\global\thin_slice\Tank409 Scan_Plains_Tank_409-_Setup_005.pcd
Tank409 Scan_Plains_Tank_409-_Setup_006 cluster 76
write to D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\target_tank\global\thin_slice\Tank409 Scan_Plains_Tank_409-_Setup_006.pcd
Tank409 Scan_Plains_Tank_409-_Setup_007 cluster 41
write to D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\target_tank\global\thin_slice\Tank409 Scan_Plains_Tank_409-_Setup_007.pcd
Tank409 Scan_Plains_Tank_409-_Setup_008 cluster 73
write to D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\target_tank\global\thin_slice\Tank409 Scan_Plains_Tank_409-_Setup_008.pcd


In [19]:
bundle.cut_horizontal(mode='target_tank', z_cut=3, thickness=0.01, write_dir=bundle.data_dir / 'target_tank' / 'global' / 'thin_slice')

Tank409 Scan_Plains_Tank_409-_Setup_004 cluster 138
write to D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\target_tank\global\thin_slice\Tank409 Scan_Plains_Tank_409-_Setup_004 cluster_138.pcd
Tank409 Scan_Plains_Tank_409-_Setup_005 cluster 63
write to D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\target_tank\global\thin_slice\Tank409 Scan_Plains_Tank_409-_Setup_005 cluster_63.pcd
Tank409 Scan_Plains_Tank_409-_Setup_006 cluster 76
write to D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\target_tank\global\thin_slice\Tank409 Scan_Plains_Tank_409-_Setup_006 cluster_76.pcd
Tank409 Scan_Plains_Tank_409-_Setup_007 cluster 41
write to D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\target_tank\global\thin_slice\Tank409 Scan_Plains_Tank_409-_Setup_007 cluster_41.pcd
Tank409 Scan_Plains_Tank_409-_Setup_008 cluster 73
write to D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\target_tank\global\thin_slice\Tank409 Scan_Plains_Tank_409-_Setup_008 cluster_73.pcd


# Align stations after 2D alignment

In [16]:
file_path = os.path.join(data_dir, 'thin_slice', 'transform_2d.pkl')
with open(file_path, 'rb') as f:
    transform_matrix = pickle.load(f)

In [ ]:
tank_id = 2
# transform the entire stations or just the target cylinder
entire_station = False

if entire_station:
    write_dir = os.path.join(data_dir, 'align_2d_station')
    if not os.path.exists(write_dir):
        os.mkdir(write_dir)
else:
    write_dir=os.path.join(data_dir, 'tank_8', 'align_2d')
    if not os.path.exists(write_dir):
        os.makedirs(write_dir)

for s, transf in zip(stations, transform_matrix):
    print(s)
    if entire_station:
        s.read_cylinders()
        # rough alignment phase 1
        s.transform(s.align['rotation'], s.align['translation'])
        # rough alignment phase 2
        s.transform(transf['rotation'], transf['translation'])
        # write
        s.write_points(os.path.join(write_dir, f'{s.name}.txt'))
    else:
        for c in s.refined_cylinders:
            try:
                # print(c.id)
                if c.tank_id == tank_id:
                    # # rough alignment phase 1
                    # c.transform(s.align['rotation'], s.align['translation'])
                    c.update_filepath(os.path.join(data_dir, 'tank_8', 'ext_ground'), num_keep_level=1, file_name=f'{s.name}.txt')
                    c.read_points()
                    # rough alignment phase 2
                    c.transform(transf['rotation'], transf['translation'])
                    # write
                    c.write_points(write_dir=write_dir, filename=f'{s.name}.txt')
                    del c.points
                    gc.collect()
            except Exception as e:
                print(e)
                pass


In [40]:
import open3d as o3d

In [ ]:
o3d.pipelines.registration.evaluate_registration